# Week 9 — Block 2: Guided Demo (Maps I)

**DATS 6401 · Visualization of Complex Data**

Real data, included in `data/` so nothing downloads:

- `states.geojson` — the 50 states, DC and Puerto Rico, from the US Census
  cartographic boundary files
- `state_stats.csv` — population and age counts from the **2024 American
  Community Survey, 5-year estimates**

Five parts:

1. Load the geometry and check the coordinate system
2. Join the statistics; counts against rates
3. Classification: how the color cuts change the map
4. Projections: the same states, three ways
5. The spatial join — matching by location instead of by key

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

states = gpd.read_file("data/states.geojson")
print(len(states), "polygons | CRS:", states.crs)
states.head(3)

## Part 1 — The coordinate system

**CRS** stands for Coordinate Reference System: the agreement about what the
numbers in the geometry mean. `EPSG:4326` means they are longitude and latitude
in degrees. `EPSG:5070` means meters in an equal-area projection built for the
United States.

Check `.crs` before anything else. The commonest geospatial bug is two layers
that look fine separately and silently fail to line up, because one is in
degrees and the other in meters.

Degrees are not a unit of length — a degree of longitude is about 111 km at the
equator and zero at the poles. So **any area or distance calculation in
EPSG:4326 is wrong**, and wrong by a different amount at every latitude.

In [ ]:
# Area computed in degrees: meaningless. Area computed in an equal-area
# projection built for the US: square kilometres you can trust.
texas = states[states["state"] == "Texas"]

bad = texas.geometry.area.iloc[0]
good = texas.to_crs(epsg=5070).geometry.area.iloc[0] / 1e6

print(f"area in EPSG:4326  : {bad:,.2f}   <- square degrees, not a real unit")
print(f"area in EPSG:5070  : {good:,.0f} km^2")
print(f"published figure   : ~696,200 km^2  (we are ~1% under: see below)")

Two things in that output are worth stopping on.

geopandas printed a **warning** when we asked for area in EPSG:4326. It knows
the answer is meaningless and says so. Read your warnings.

The projected figure comes out about 1% below the published one, and that is not
an error in the projection — it is the **boundary file**. These outlines have
been simplified so the file stays small, which smooths away inlets and river
bends and therefore loses a little area. Generalized boundaries are fine for a
choropleth, where you are coloring regions, and wrong for measurement. Pick the
resolution to match the question.

## Part 2 — Counts against rates

The join here is an ordinary `merge` on the state FIPS code — a plain attribute
join, no geography involved. Part 5 does the version that joins on location.

In [ ]:
stats = pd.read_csv("data/state_stats.csv", dtype={"fips": str})
us = states.merge(stats, on="fips", suffixes=("_geo", ""))
print(len(us), "states joined")

# Alaska, Hawaii and Puerto Rico are far from the rest, so a single map of all
# 52 is mostly empty ocean. Dropping them is a choice, and it belongs in the
# caption rather than being done quietly.
conus = us[~us["state"].isin(["Alaska", "Hawaii", "Puerto Rico"])]
us[["state", "pop_total", "pop_65plus", "pct_65plus"]].head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
conus.plot(column="pop_65plus", cmap="OrRd", legend=True, ax=axes[0],
           edgecolor="white", linewidth=0.4)
axes[0].set_title("Count: people aged 65+")
conus.plot(column="pct_65plus", cmap="OrRd", legend=True, ax=axes[1],
           edgecolor="white", linewidth=0.4)
axes[1].set_title("Share: % of the state aged 65+")
for a in axes: a.axis("off")
plt.suptitle("2024 ACS 5-year estimates, 49 states and DC")
plt.tight_layout(); plt.show()

In [ ]:
ca = conus[conus["state"] == "California"].iloc[0]
n = len(conus)
print(f"California: {int(ca.pop_65plus):,} people aged 65+, {ca.pct_65plus}% of the state")
print(f"  rank by count: {int((conus.pop_65plus > ca.pop_65plus).sum()) + 1} of {n}")
print(f"  rank by share: {int((conus.pct_65plus > ca.pct_65plus).sum()) + 1} of {n}")
print()
print("top three by count:", ", ".join(conus.nlargest(3, "pop_65plus")["state"]))
print("top three by share:", ", ".join(conus.nlargest(3, "pct_65plus")["state"]))

**Say this out loud.** California is first by count and 44th of 49 by share. The
count map is not showing where older people live; it is showing where people
live, because a populous state has more of everything.

Map the rate unless the count is the thing you are actually asking about. A
count answers "how many ambulances do we need"; a rate answers "is this place
unusual".

## Part 3 — Classification

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, scheme in zip(axes, ["quantiles", "equal_interval"]):
    conus.plot(column="pct_65plus", scheme=scheme, k=5, cmap="OrRd", legend=True,
               ax=ax, edgecolor="white", linewidth=0.4,
               legend_kwds={"fontsize": 7, "loc": "lower left"})
    ax.set_title(f'scheme="{scheme}", k=5'); ax.axis("off")
plt.suptitle("The same numbers, cut two different ways")
plt.tight_layout(); plt.show()

**Quantiles** puts an equal number of states in each class, so every color gets
used. **Equal interval** cuts the range into five equal-width slices, so a
bunched-up distribution lands mostly in one class.

Neither is wrong. What is wrong is not saying which one you used — the reader
cannot tell from the picture, and the two pictures disagree.

## Part 4 — Projections

In [ ]:
# The Earth is round and the slide is flat, so something has to give. Each of
# these keeps one property and sacrifices the others.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (epsg, label) in zip(axes, [
        (4326, "EPSG:4326 lon/lat\nnot really a projection"),
        (3857, "EPSG:3857 Web Mercator\nshapes kept, areas inflated north"),
        (5070, "EPSG:5070 Albers equal-area\nareas comparable")]):
    conus.to_crs(epsg=epsg).plot(ax=ax, color="#EAF2F6", edgecolor="#2E6E8E",
                                 linewidth=0.4)
    ax.set_title(label, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

Compare the width of the northern states across the three. Web Mercator is what
almost every web map uses, and it stretches everything far from the equator —
which is why a choropleth of areas should not be drawn in it.

## Part 5 — The spatial join

In [ ]:
import numpy as np
from shapely.geometry import Point

# These points are RANDOM, not real observations. They exist to demonstrate the
# mechanics of a spatial join; nothing should be concluded from where they fall.
rng = np.random.default_rng(7)
pts = gpd.GeoDataFrame(
    geometry=[Point(rng.uniform(-124, -67), rng.uniform(25, 49)) for _ in range(300)],
    crs="EPSG:4326")

# sjoin is merge where the key is geometry rather than a column.
tagged = gpd.sjoin(pts, conus[["state", "geometry"]], predicate="within")
print(f"{len(pts)} points, {len(tagged)} landed inside a state "
      f"({len(pts) - len(tagged)} fell in the sea or over Canada or Mexico)")
print(tagged.groupby("state").size().sort_values(ascending=False).head(5))

fig, ax = plt.subplots(figsize=(8, 4.5))
conus.plot(ax=ax, color="#EAF2F6", edgecolor="#2E6E8E", linewidth=0.4)
pts.plot(ax=ax, color="#d9534f", markersize=5)
ax.set_title("300 random points, tagged with the state they fall in"); ax.axis("off")
plt.tight_layout(); plt.show()

## Part 6 — The same map, interactive

Everything so far has been a picture. One line turns it into a map you can pan,
zoom and hover: `.explore()` builds a **Leaflet** map — the library behind most
web maps — using **folium** underneath.

This needs `folium` and `mapclassify` installed, and it needs internet, because
the Leaflet library and the map tiles are fetched from the web when the map is
displayed.

In [ ]:
# pip install folium mapclassify
conus.explore(
    column="pct_65plus",
    cmap="OrRd",
    scheme="quantiles", k=5,
    tooltip=["state", "pct_65plus", "pop_65plus"],
    legend=True,
    tiles=None,              # no basemap: the states carry the data
)

**Hovering is the thing the static map cannot do.** It puts the number next to
the color, so a reader does not have to measure a shade against a legend — and
measuring color against a legend is exactly what people are bad at.

What it does not fix: the classification scheme is still a choice, the
projection is still a choice, and three places are still missing. Interactivity
makes a map easier to read. It does not make it honest.

`app_map_starter.py` beside this notebook is the same idea as a Streamlit app,
with the measure and the scheme as controls so you can watch the map change.

`sjoin(points, polygons, predicate="within")` is a `merge` whose key is
geometry. It requires both layers in the **same CRS** — which is where this part
of the course always ends up.

Notice that points outside every polygon are simply dropped. If you had counted
rows after the join and assumed nothing was lost, you would be short by the
number that fell in the ocean.

## Wrap-up → Block 3

**read → check CRS → join → rate → disclose the scheme → project → sjoin.**
Block 3 runs the same pipeline on a column you choose.